# Overfitting, Underfitting & Cross-Validation
**Level:** Beginner | **Interactive Machine Learning Curriculum**

> The Bias-Variance tradeoff, K-Fold cross validation, Stratified K-Fold, Time-Series splits, nested cross-validation, and preventing data leakage.

---

### 📖 Conceptual Foundations

#### The Bias-Variance Tradeoff

The expected generalization test error of any machine learning model decomposes into three mathematically distinct components:

$$\text{Expected Test Error} = \text{Bias}^2 + \text{Variance} + \text{Irreducible Error} (\sigma^2)$$

- **Bias (Underfitting):** Error stemming from overly simplistic assumptions (e.g. fitting a straight line to quadratic data). High train error AND high test error.
- **Variance (Overfitting):** Error stemming from excessive model sensitivity to training noise (e.g. unconstrained decision tree). Low train error BUT high test error.
- **Irreducible Error ($\sigma^2$):** Inherent stochastic noise in the real-world data generation process that cannot be eliminated by any model.


#### Cross-Validation Architectures

A simple train/test split can be noisy if the test set is small. **K-Fold Cross-Validation** provides a robust statistical estimate of performance across the entire dataset:

1. Partition the dataset into $K$ equal-sized folds.
2. For each fold $k \in \{1, \dots, K\}$:
   - Train on the remaining $K-1$ folds.
   - Evaluate performance on fold $k$.
3. Average the evaluation scores across all $K$ iterations.


#### Cross-Validation Strategies Comparison

| Strategy | Mechanism | When to Use |
|---|---|---|
| **Standard K-Fold** | Randomly partitions data into $K$ disjoint folds | Balanced regression or balanced classification datasets |
| **Stratified K-Fold** | Preserves identical class percentage distribution in every fold | **Mandatory for imbalanced classification** (e.g. fraud detection) |
| **Time-Series Split** | Forward chaining (train on past, test on future; never shuffle) | **Mandatory for temporal / time-series data** to prevent lookahead leakage |
| **Group K-Fold** | Ensures samples from the same group/patient stay in the same fold | Grouped data (e.g., multiple X-rays from the same medical patient) |
| **Leave-One-Out (LOO)** | $K = N$ (train on $N-1$, test on 1 sample) | Extremely small datasets ($N < 50$), computationally heavy |


#### The #1 Pipeline Rule to Prevent Data Leakage

**Never fit any preprocessor on the whole dataset!**
All scalers, encoders, and imputers must be fitted **exclusively on the training folds (`fit_transform`)**, and only applied to test folds via **`transform`**. Use Scikit-learn `Pipeline` objects to guarantee this programmatically!


#### Key Validation Rule

Your validation setup must faithfully mimic how the model will be deployed in the real world. If data arrives sequentially over time, use a temporal walk-forward split, never random shuffling!



In [ ]:
# %matplotlib inline
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# Load dataset
data = load_breast_cancer()
X, y = data.data, data.target

# Construct Pipeline: Scaler is fitted ONLY inside training folds
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(random_state=42, max_iter=1000))
])

# Configure 5-Fold Stratified Cross-Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(pipeline, X, y, cv=skf, scoring='roc_auc')

print('=== 5-Fold Stratified Cross-Validation Results ===')
for fold, score in enumerate(scores, 1):
    print(f'Fold {fold}: ROC-AUC = {score:.4f}')
print(f'\nMean ROC-AUC: {np.mean(scores):.4f} (+/- {np.std(scores):.4f})')


---

### 🎯 Hands-On Experiments & Challenges
1. **Experiment with parameters:** Modify the hyperparameters or dataset sizes in the cell above.
2. **Add visual diagnostics:** Plot loss curves or feature distributions using Matplotlib/Seaborn.
3. **Test with new data:** Try running the model on synthetic noisy data or custom inputs.


In [ ]:
# Your custom experiment code here:

